Block 1: Symmetry Block (Initial Setup and Small-k Screening)

In [46]:
import sympy
from math import gcd
import random

def extract_digits(n, base):
    """Extract digits of n in given base (O(log n) passive)."""
    if n == 0:
        return [0]
    digits = []
    while n > 0:
        digits.append(n % base)
        n //= base
    return digits[::-1]  # MSB first

def symmetry_block(n, k):
    """Symmetry Block: Trust C(n,k) == C(n, n-k); infer from mirrored digits/parity."""
    mirrored_k = n - k
    # Trust law: If parity mismatch in indices, infer asymmetry (potential leak)
    if (k % 2) != (mirrored_k % 2):
        return True  # Trust asymmetry -> leak hint
    # Free: Digit length match (for base 10 approx, but positional)
    digits_n = len(str(n))
    digits_k = len(str(k))
    digits_mir = len(str(mirrored_k))
    if digits_k != digits_mir:
        return True  # Mismatch -> disruption
    return False  # Symmetric clean

# Placeholder for full assembly; will build upon in later blocks
def factor_semiprime_via_laws(n):
    """Main function - builds with each block."""
    if sympy.isprime(n):
        return None
    # Small-k screening with symmetry
    anchors = [2,3,5,7,11,13]
    for k in anchors:
        if k >= n:
            break
        if symmetry_block(n, k):
            # Trust leak; hypothesize small p=k
            if n % k == 0:
                p, q = k, n // k
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    # Escalation stub for later blocks
    return None

# Demo stub
print(factor_semiprime_via_laws(15))  # Expected [3,5]

[3, 5]


Block 2: Recursion Block (Escalation Guidance)

In [47]:
def recursion_block(n, k):
    """Recursion Block: Trust C(n,k) = C(n-1,k-1) + C(n-1,k); infer chain from edges."""
    # Trust: If k or n-k is edge (0 or 1 or n-1), infer propagated zero unless disruption
    if k <= 1 or k >= n - 1:
        return False  # Edge clean
    # Free: Adjacency parity - if both priors even parity, trust zero chain
    prior1 = k - 1
    prior2 = k
    if (prior1 % 2 == 0) and (prior2 % 2 == 0):
        return False  # Trust propagated zero
    # Disruption if odd chain break
    return True  # Trust sum break -> potential leak

# Updated main: Integrate recursion in escalation
def factor_semiprime_via_laws(n):
    """Main function - builds with each block."""
    if sympy.isprime(n):
        return None
    # Small-k screening with symmetry
    anchors = [2,3,5,7,11,13]
    for k in anchors:
        if k >= n:
            break
        if symmetry_block(n, k):
            # Trust leak; hypothesize small p=k
            if n % k == 0:
                p, q = k, n // k
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    # Escalation with recursion
    sqrt_n = int(n**0.5) + 1
    for k in range(14, sqrt_n, 2):  # Step 2 for parity
        if recursion_block(n, k):
            # Trust disruption; check if factor
            if n % k == 0:
                p, q = k, n // k
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    return None

# Demo
print(factor_semiprime_via_laws(91))  # 7*13=91

[7, 13]


Block 3: Digit Subset Block (Integrated Screening and Escalation)

In [48]:
def digit_subset_block(n, k, base=10):  # Approx base for digits
    """Digit Subset Block: Trust Lucas - leak if k digits not subset of n's."""
    digits_n = extract_digits(n, base)
    digits_k = extract_digits(k, base)
    # Pad shorter to match length
    max_len = max(len(digits_n), len(digits_k))
    digits_n += [0] * (max_len - len(digits_n))
    digits_k += [0] * (max_len - len(digits_k))
    # Trust: If any k_i > n_i, violation -> leak
    for i in range(max_len):
        if digits_k[i] > digits_n[i]:
            return True  # Trust subset violation -> leak
    return False  # Subset holds -> clean

# Updated main: Integrate digit subset in screening/escalation
def factor_semiprime_via_laws(n):
    """Main function - builds with each block."""
    if sympy.isprime(n):
        return None
    base = 10  # Fixed for demo
    # Small-k screening with symmetry + digit subset
    anchors = [2,3,5,7,11,13]
    for k in anchors:
        if k >= n:
            break
        if symmetry_block(n, k) or digit_subset_block(n, k, base):
            # Trust leak; hypothesize p=k
            if n % k == 0:
                p, q = k, n // k
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    # Escalation with recursion + digit subset
    sqrt_n = int(n**0.5) + 1
    for k in range(14, sqrt_n, 2):
        if recursion_block(n, k) or digit_subset_block(n, k, base):
            if n % k == 0:
                p, q = k, n // k
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    return None

# Demo
print(factor_semiprime_via_laws(323))  # 17*19=323

None


Block 4: Carry Count Block (Disruption Pinpointing)

In [49]:
def carry_count_block(n, k, base=10):
    """Carry Count Block: Trust Kummer - count carries in k + (n-k) base base."""
    nk = n - k
    digits_k = extract_digits(k, base)
    digits_nk = extract_digits(nk, base)
    max_len = max(len(digits_k), len(digits_nk))
    digits_k += [0] * (max_len - len(digits_k))
    digits_nk += [0] * (max_len - len(digits_nk))
    carry_count = 0
    carry = 0
    for i in range(max_len):
        total = digits_k[i] + digits_nk[i] + carry
        if total >= base:
            carry_count += 1
            carry = 1
        else:
            carry = 0
    # Trust: If carry_count ==1, leak for semiprime valuation
    return carry_count == 1

# Updated main: Integrate carry in escalation/disruption
def factor_semiprime_via_laws(n):
    """Main function - builds with each block."""
    if sympy.isprime(n):
        return None
    base = 10
    # Small-k screening with symmetry + digit subset
    anchors = [2,3,5,7,11,13]
    for k in anchors:
        if k >= n:
            break
        if symmetry_block(n, k) or digit_subset_block(n, k, base):
            # Trust leak; hypothesize p=k
            if n % k == 0:
                p, q = k, n // k
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    # Escalation with recursion + digit subset + carry
    sqrt_n = int(n**0.5) + 1
    for k in range(14, sqrt_n, 2):
        if recursion_block(n, k) or digit_subset_block(n, k, base) or carry_count_block(n, k, base):
            if n % k == 0:
                p, q = k, n // k
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    return None

# Demo
print(factor_semiprime_via_laws(10007*10009))  # Large small demo ~10^8

None


Block 5: Fractal Pattern Block (Final Assembly and Extraction)

In [50]:
def fractal_pattern_block(n, k):
    """Fractal Pattern Block: Trust Sierpinski mod 2 - disruption if not subset bits."""
    # Trust bitwise: n & k != k -> not subset, potential disruption
    if (n & k) != k:
        return True  # Trust non-subset -> leak/disruption
    # For mod 3 approx, simple parity double-check
    if bin(k).count('1') % 3 != bin(n).count('1') % 3:  # Rough fractal echo
        return True
    return False  # Subset holds -> clean pattern

# Final main: Integrate fractal in all, step=1 for general, full assembly
def factor_semiprime_via_laws(n):
    """Main function - complete assembly trusting all blocks."""
    if sympy.isprime(n):
        return None
    base = 10
    # Small-k screening with all blocks
    anchors = [2,3,5,7,11,13]
    for k in anchors:
        if k >= n:
            break
        if (symmetry_block(n, k) or
            recursion_block(n, k) or
            digit_subset_block(n, k, base) or
            carry_count_block(n, k, base) or
            fractal_pattern_block(n, k)):
            if n % k == 0:
                p, q = k, n // k
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    # Escalation with all blocks chained
    sqrt_n = int(n**0.5) + 1
    for k in range(14, sqrt_n):  # Step 1 for general
        if (symmetry_block(n, k) or
            recursion_block(n, k) or
            digit_subset_block(n, k, base) or
            carry_count_block(n, k, base) or
            fractal_pattern_block(n, k)):
            if n % k == 0:
                p, q = k, n // k
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    return None

# Final demo: Test with previous and new
print(factor_semiprime_via_laws(10007*10009))  # Now with step=1, full chain
print(factor_semiprime_via_laws(17*19))  # Quick retest

[10007, 10009]
[17, 19]


Stricter Alignment + Binary Search Setup

In [51]:
def hockey_stick_cumulative_block(n, m, r=1):
    """Hockey-Stick Cumulative Block: Trust sum_{i=r}^n C(i,r) = C(n+1, r+1); infer mid disruption scale."""
    # Trust law: For small r, sum up to m ~ C(m+1, r+1) mod n; if parity of m/r mismatches n's, infer disruption before/after mid
    if m % r != n % r:  # Positional mismatch in residue
        return True  # Trust cumulative leak before mid -> search left
    # Free: Digit length of m vs n for scale estimate
    if len(str(m)) != len(str(n)) // 2:
        return False  # Uniform scale -> no disruption, search right
    return True  # Disruption at mid scale

# Updated main with AND + binary search guidance
def factor_semiprime_via_laws(n):
    """Refined assembly: AND blocks + binary search via hockey-stick."""
    if sympy.isprime(n):
        return None
    base = 10
    anchors = [2,3,5,7,11,13]
    for k in anchors:
        if k >= n:
            break
        if (symmetry_block(n, k) and
            recursion_block(n, k) and
            digit_subset_block(n, k, base) and
            carry_count_block(n, k, base) and
            fractal_pattern_block(n, k)):
            if n % k == 0:
                p, q = k, n // k
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    # Binary search escalation guided by hockey-stick
    low, high = 14, int(n**0.5) + 1
    while low <= high:
        mid = (low + high) // 2
        if hockey_stick_cumulative_block(n, mid):
            # Trust disruption left of mid -> search left
            high = mid - 1
            # Check mid as candidate
            if (symmetry_block(n, mid) and
                recursion_block(n, mid) and
                digit_subset_block(n, mid, base) and
                carry_count_block(n, mid, base) and
                fractal_pattern_block(n, mid)):
                if n % mid == 0:
                    p, q = mid, n // mid
                    if sympy.isprime(p) and sympy.isprime(q):
                        return sorted([p, q])
        else:
            low = mid + 1
    return None

Combinatorial Sum Block (Residue Sums for Global Escalation)

In [52]:
def combinatorial_sum_block(n, r=1, i=0, a=1):
    """Combinatorial Sum Block: Trust sum_{k≡i mod r} C(n,k) a^{n-k} gcd non-trivial with n for r ≤ FAC ≤ p."""
    # Trust law: For small r, sum has gcd >1 with n if r factorization-friendly; infer from residue digits
    # Free positional: Check if n digits mod r mismatch i (residue violation -> leak/gcd hint)
    res_n = n % r
    if res_n != i % r:
        return True  # Trust non-trivial gcd hint -> factor at scale r ≈ p/r or something; for demo, flag for % probe at k=r
    # Additional: Symmetry in residues
    if (n % (2*r)) % 2 != 0:  # Odd residue disruption
        return True
    return False  # Clean sum -> no factor

# Integrate into main: Use for r-escalation in binary search (probe r instead of k for global sums)
def factor_semiprime_via_laws(n):
    """Full refined: AND + binary + combinatorial sum for r-probes."""
    if sympy.isprime(n):
        return None
    base = 10
    anchors = [2,3,5,7,11,13]
    for k in anchors:
        if k >= n:
            break
        if (symmetry_block(n, k) and
            recursion_block(n, k) and
            digit_subset_block(n, k, base) and
            carry_count_block(n, k, base) and
            fractal_pattern_block(n, k)):
            if n % k == 0:
                p, q = k, n // q
                if sympy.isprime(p) and sympy.isprime(q):
                    return sorted([p, q])
    # r-escalation with combinatorial sum + binary on r scale (trust FAC small)
    low_r, high_r = 2, min(1000, int(n**0.25))  # Small r up to n^{1/4} for balanced
    while low_r <= high_r:
        mid_r = (low_r + high_r) // 2
        if combinatorial_sum_block(n, mid_r):
            # Trust gcd leak at r-scale -> probe k near mid_r * log or something; for demo, check k=mid_r
            k = mid_r
            if (symmetry_block(n, k) and
                recursion_block(n, k) and
                digit_subset_block(n, k, base) and
                carry_count_block(n, k, base) and
                fractal_pattern_block(n, k)):
                if n % k == 0:
                    p, q = k, n // k
                    if sympy.isprime(p) and sympy.isprime(q):
                        return sorted([p, q])
            high_r = mid_r - 1  # Search smaller r
        else:
            low_r = mid_r + 1
    # Fallback binary on k if no sum leak
    low, high = 14, int(n**0.5) + 1
    while low <= high:
        mid = (low + high) // 2
        if hockey_stick_cumulative_block(n, mid):
            high = mid - 1
            k = mid
            if (symmetry_block(n, k) and
                recursion_block(n, k) and
                digit_subset_block(n, k, base) and
                carry_count_block(n, k, base) and
                fractal_pattern_block(n, k)):
                if n % k == 0:
                    p, q = k, n // k
                    if sympy.isprime(p) and sympy.isprime(q):
                        return sorted([p, q])
        else:
            low = mid + 1
    return None

Block 6: Benchmark Block (Comprehensive Test on 10^6 to 10^18)

In [53]:
import time

def benchmark_block():
    """Benchmark Block: Comprehensive test on semiprimes 10^6 to 10^18 using known primes list."""
    # Hardcode small odd primes for low ranges
    small_primes = [3,5,7,11,13,17,19,23,29,31,37,41,43,47,53,59,61,67,71,73,79,83,89,97,101,103,107,109,113,127,131,137,139,149,151,157,163,167,173,179,181,191,193,197,199]

    # Provided list as known primes p_n
    primes_data = [
        (50000, 611953), (60000, 746773), (70000, 882377),
        (80000,1020379), (90000,1159523), (100000,1299709),
        (110000,1441049), (120000,1583539), (130000,1726943),
        (140000,1870667),
        (500000, 7368787), (600000, 8960453), (700000,10570841),
        (800000,12195257), (900000,13834103), (1000000,15485863),
        (1100000,17144489), (1200000,18815231), (1300000,20495843),
        (1400000,22182343),
        (5000000, 86028121), (6000000,104395301), (7000000,122949823),
        (8000000,141650939), (9000000,160481183), (10000000,179424673),
        (11000000,198491317), (12000000,217645177), (13000000,236887691),
        (14000000,256203161),
        (50000000, 982451653), (60000000,1190494759), (70000000,1400305337),
        (80000000,1611623773), (90000000,1824261409), (100000000,2038074743),
        (110000000,2252945251), (120000000,2468776129), (130000000,2685457421),
        (140000000,2902958801),
        (1000000000, 22801763489), (10000000000, 252097800623),
        (100000000000, 2760727302517), (1000000000000, 29996224275833),
        (10000000000000, 323780508946331), (100000000000000, 3475385758524527),
        (1000000000000000, 37124508045065437), (10000000000000000, 394906913903735329),
        (100000000000000000, 4185296581467695669), (1000000000000000000, 44211790234832169331)
    ]

    known_primes = [p for _, p in primes_data]
    all_primes = small_primes + known_primes  # Combined list

    # Generate 12 sampled semiprimes across ranges (one per decade, varying small p for scaling test)
    semiprimes = []
    for exp in range(6, 18):  # 10^6 to 10^17
        low = 10**exp
        high = 10**(exp + 1)
        # Vary small p index for scaling (larger p for larger exp)
        i_start = max(0, (exp - 6) * 2)  # Increase i with exp for larger small p
        found = False
        for i in range(i_start, min(i_start + 5, len(all_primes) - 1)):  # Try 5 candidates
            p = all_primes[i]
            for j in range(i + 1, len(all_primes)):
                q = all_primes[j]
                n = p * q
                if low <= n < high:
                    semiprimes.append((n, sorted([p, q])))
                    found = True
                    break
            if found:
                break
        if len(semiprimes) >= 12:
            break

    # Run benchmark
    results = []
    for n, true_factors in semiprimes:
        start = time.time()
        found = factor_semiprime_via_laws(n)
        end = time.time()
        success = found == true_factors if found else False
        results.append({
            'log10_n': len(str(n)) - 1,
            'n': n,
            'time_s': end - start,
            'success': success,
            'small_factor': true_factors[0]
        })

    # Summary stats
    total = len(results)
    success_rate = sum(1 for r in results if r['success']) / total * 100 if total > 0 else 0
    avg_time = sum(r['time_s'] for r in results) / total if total > 0 else 0
    print(f"Comprehensive Benchmark: {total} semiprimes from 10^6 to 10^18 (sampled, no primality tests).")
    print(f"Success Rate: {success_rate:.2f}% | Avg Time: {avg_time:.6f}s | Scaling Note: Time ~ O(small p size)")
    for r in results:
        print(f"10^{r['log10_n']}: n={r['n']}, Success={r['success']}, Time={r['time_s']:.6f}s, Small p={r['small_factor']}")

    return results

# Run the benchmark (complete script execution)
if __name__ == "__main__":
    # Re-run benchmark with refinements (add balanced example)
    balanced_n = 10007 * 10009
    start = time.time()
    balanced_found = factor_semiprime_via_laws(balanced_n)
    end = time.time()
    print(f"Balanced 10^8: {balanced_n} = {balanced_found}, time = {end - start:.6f}s (True: [10007, 10009])")
    # Full benchmark
    benchmark_block()


Balanced 10^8: 100160063 = None, time = 0.000377s (True: [10007, 10009])
Comprehensive Benchmark: 12 semiprimes from 10^6 to 10^18 (sampled, no primality tests).
Success Rate: 0.00% | Avg Time: 0.000279s | Scaling Note: Time ~ O(small p size)
10^6: n=1835859, Success=False, Time=0.000199s, Small p=3
10^7: n=10087343, Success=False, Time=0.000139s, Small p=7
10^8: n=116485889, Success=False, Time=0.000175s, Small p=13
10^9: n=1634534299, Success=False, Time=0.000204s, Small p=19
10^10: n=28491097937, Success=False, Time=0.000226s, Small p=29
10^11: n=107409475637, Success=False, Time=0.000314s, Small p=37
10^12: n=1071682883983, Success=False, Time=0.000289s, Small p=47
10^13: n=13361183433019, Success=False, Time=0.000366s, Small p=53
10^14: n=168404365453537, Success=False, Time=0.000384s, Small p=61
10^15: n=2129731923584143, Success=False, Time=0.000386s, Small p=71
10^16: n=25578660206760149, Success=False, Time=0.000327s, Small p=79
10^17: n=309309332508682903, Success=False, Time